# Lab 2: Apache Spark — analiza danych transakcyjnych

In [50]:
from pyspark.sql import SparkSession

In [51]:
# from pyspark import SparkContext
# jak byśmy zobaczyli coś takiego, to bardzo możliwe, że bazujemy na dość starej wersji
# można uruchomić obie, ale równocześnie SparkSession zawiera w sobie to co SparkCOntext

In [52]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

In [53]:
spark

In [54]:
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [55]:
# tutaj dodatkowo znajduję gdzie jest zlokalizowany plik, którego szukam
!find /home/jovyan -name "transactions_10k.jsonl"

/home/jovyan/notebooks/ADwCR/lab2/cwiczenia/RTA/transactions_10k.jsonl
/home/jovyan/notebooks/ADwCR/lab2/transactions_10k.jsonl


In [56]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [57]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [58]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



# Zadanie 2.1 — Liczba transakcji i suma przychodów per sklep

In [59]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store") #pogrupuj po sklepie
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    #tutaj agregujemy i warto dodać aliasy, bo inaczej nazwy wygeneruje automatycznie i różnie można trafić
    .orderBy("store")
)

# ta kamórka wykonuje się bardzo szybko, bo to jedynie definiuje co ma się wydarzyć, jeżeli zostanie wywołane 
# coś związane ze store_summary

In [60]:
store_summary.show()
# dopiero tutaj zajmuje to chwilę czasu, bo coś chcemy wywołać 

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



# Zadanie 2.2 - Statystyki per kategoria

In [62]:
from pyspark.sql.functions import min as _min, max as _max

# MÓJ KOD
category_stats = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

category_stats.show()

+-----------+---------+-----------+-------+-------+
|   category|liczba_tx|srednia_PLN|min_PLN|max_PLN|
+-----------+---------+-----------+-------+-------+
|elektronika|     2542|     598.26|    9.0| 9999.0|
|    książki|     2574|     330.76|    5.0|9107.25|
|     odzież|     2453|     346.46|    5.0|9696.63|
|    żywność|     2431|     324.77|    5.0|6916.92|
+-----------+---------+-----------+-------+-------+



# Zadanie 3.1 — Liczba transakcji per godzina (tumbling 1h)

In [63]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    # czyli teraz nie po kolumnie, tylko po oknie
    # Okno (window) dzieli oś czasu na przedziały i grupuje dane osobno w każdym.
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)

In [64]:
hourly.show(truncate=False)
# truncate=False po to, żeby rozszerzyć widoczność (i przejrzystość) okna

# tu w ramce mamy kolumny: okno i agregaty
# tu ewidentnie widzimy, że to ramka, bo są nawiasy z wąsami

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [65]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [66]:
# mieliśmy 3 kolumny (z czego pierwsza składała się z dwóch tak jakby, które chcemy teraz rozdzielić)
# teraz tworzymy nowy widok, żeby mieć 4 kolumny

(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



# Zadanie 3.2 Okna 30-minutowe per sklep
Policz transakcje i sumę per sklep w każdym 30-minutowym oknie. Posortuj po oknie, a w ramach okna po sklepie.

In [67]:
# MÓJ KOD

from pyspark.sql.functions import sum

df.groupBy(
    window("timestamp", "30 minutes"),
    "store"
).agg(
    count("tx_id").alias("liczba_tx"),
    _round(sum("amount"), 2).alias("suma_PLN")
).orderBy("window").show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Warszawa|584      |214573.66|
|{2026-04-12 09:00:00, 2026-04-12 09:3

# Zadanie 3.3 — W której godzinie sklep “Kraków” miał najwyższy przychód?
Filtruj najpierw po sklepie, potem zrób okno godzinne, posortuj malejąco po sumie.

In [68]:
from pyspark.sql.functions import desc

# MÓJ KOD

from pyspark.sql.functions import desc

krakow_top_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN"
    )
    .orderBy(desc("suma_PLN"))
)

krakow_top_hour.show(truncate=False)

+-------------------+-------------------+---------+---------+
|od                 |do                 |liczba_tx|suma_PLN |
+-------------------+-------------------+---------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169     |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|821      |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|532      |201259.26|
+-------------------+-------------------+---------+---------+



# Zadanie 4.1 — Okno 1h, krok 30 minut

In [69]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



# Zadanie 4.2 — Porównaj tumbling vs sliding
Policz łączną liczbę wierszy wynikowych w obu podejściach. Dlaczego sliding daje więcej wierszy?

In [70]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


## Dlaczego sliding ma więcej wierszy?
Sliding window daje więcej wyników, ponieważ okna nachodzą na siebie. 
Jedna transakcja może należeć do wielu okien czasowych.
Tumbling window tworzy rozłączne przedziały czasu, więc każde zdarzenie trafia tylko do jednego okna.

# Część 5: Pytania kontrolne

### 1. Ile transakcji jest w oknie 09:00–10:00?
4661

### 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
groupBy("store") grupuje dane tylko według sklepu, czyli pokazuje łączny wynik dla każdego sklepu w całym zbiorze danych. Natomiast groupBy(window(...), "store") grupuje dane jednocześnie według sklepu i przedziału czasowego, więc pokazuje wyniki sklepów osobno w kolejnych oknach czasu.

### 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
2 okna: 09:00–10:00 oraz 09:30–10:30.

# PRACA DOMOWA

## 1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

In [74]:
from pyspark.sql.functions import avg, asc

gdansk_lowest_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN"),
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_PLN",
        "liczba_tx"
    )
    .orderBy(asc("srednia_PLN"))
)

gdansk_lowest_avg.show(truncate=False)

+-------------------+-------------------+-----------+---------+
|od                 |do                 |srednia_PLN|liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |766      |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92     |558      |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91     |1174     |
+-------------------+-------------------+-----------+---------+



### Odpowiedź

Najniższa średnia kwota transakcji w sklepie Gdańsk wystąpiła w godzinie 08:00–09:00.

## 2. Policz ile transakcji per kategoria było w oknie 09:00–09:30.

In [75]:
category_0900_0930 = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") < "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy("category")
)

category_0900_0930.show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



## 3. Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

In [76]:
szczyt_transakcji_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(desc("liczba_tx"))
)

szczyt_transakcji_15min.show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+



### Odpowiedź
Szczyt transakcji wystąpił w oknie 09:15–09:30.